# Parsing the kcat values predicted by turnup

In [1]:
import os
import numpy as np
import cobra
import pandas as pd
import numpy as np

from PAModelpy.utils.pam_generation import merge_enzyme_complexes, _get_genes_for_proteins

if os.path.split(os.getcwd())[1]=='Yeast9':
    os.chdir('../../..')

Loading PAModelpy modules version 0.0.4.2


In [2]:
TURNUP_KCAT_FILE = os.path.join('Data', 'turnup_kcat_predictions', 'Tobias', 'Yeast9_predicted_kcats.xlsx')
BIGG_YEAST_MAPPING_FILE = os.path.join('Data', 'turnup_kcat_predictions', 'Tobias', 'Yeast_9_0_1_xba_mapping.xlsx')
YEAST_MODEL_FILE = os.path.join('Models', 'yeast9.xml')
UNIPROT_FILE_PATH = os.path.join('Data', 'Databases','uniprotkb_scerevisiae_240903.xlsx')
OUTPUT_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_yeast9_EnzymaticData_TurnUp.xlsx')
GotEnzymes_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_yeast9_EnzymaticData_240903.xlsx')

#You need to adjust this to find the geneid/locustag for your microbe
locustag_regex =r'(Y[A-P][LR][0-9]{3}[CW])'

## Extract the TurnUp kcats and map to the model ids

In [3]:
turnup_kcats = pd.read_excel(TURNUP_KCAT_FILE, index_col=0)
bigg2yeast_mapping = pd.read_excel(BIGG_YEAST_MAPPING_FILE, sheet_name='orig2new_rid')
bigg2yeast_mapping.columns = ['model_rid','rid']

turnup_kcats = turnup_kcats.merge(bigg2yeast_mapping, on='rid')
turnup_kcats['model_rid'] = turnup_kcats['model_rid'].str.removeprefix('R_')
turnup_kcats

,rid,dirxn,enzyme,kcat_per_s,notes,info_active_sites,info_ecns,info_type,info_genes,info_name,info_reaction,model_rid
0,R_13BGHe,1,enz_YDR261C,13.67,TurNuP predicted,1,"3.2.1.58, 3.2.1.6",metabolic,YDR261C,"exo-1,3-beta-glucan glucohydrase",M_13BDglcn_e + M_h2o_e => M_glc__D_e,r_0370
1,R_13BGHe,1,enz_YGR282C,27.81,TurNuP predicted,1,"3.2.1.58, 3.2.1.6",metabolic,YGR282C,"exo-1,3-beta-glucan glucohydrase",M_13BDglcn_e + M_h2o_e => M_glc__D_e,r_0370
2,R_13BGHe,1,enz_YLR300W,25.83,TurNuP predicted,1,"3.2.1.58, 3.2.1.6",metabolic,YLR300W,"exo-1,3-beta-glucan glucohydrase",M_13BDglcn_e + M_h2o_e => M_glc__D_e,r_0370
3,R_13BGHe,1,enz_YNR067C,24.95,TurNuP predicted,1,"3.2.1.58, 3.2.1.6",metabolic,YNR067C,"exo-1,3-beta-glucan glucohydrase",M_13BDglcn_e + M_h2o_e => M_glc__D_e,r_0370
4,R_13BGHe,1,enz_YOR190W,24.61,TurNuP predicted,1,"3.2.1.58, 3.2.1.6",metabolic,YOR190W,"exo-1,3-beta-glucan glucohydrase",M_13BDglcn_e + M_h2o_e => M_glc__D_e,r_0370
...,...,...,...,...,...,...,...,...,...,...,...,...
4888,R_r_4777,1,enz_YKL067W,23.92,TurNuP predicted,6,2.7.4.6,metabolic,YKL067W,nucleoside diphosphate kinase,M_atp_m + M_didp_m -> M_adp_m + M_ditp_m,r_4777
4889,R_r_4777,-1,enz_YKL067W,23.92,TurNuP predicted,6,2.7.4.6,metabolic,YKL067W,nucleoside diphosphate kinase,M_adp_m + M_ditp_m -> M_atp_m + M_didp_m,r_4777
4890,R_r_4778,1,enz_YKL067W,23.92,TurNuP predicted,6,2.7.4.6,metabolic,YKL067W,nucleoside diphosphate kinase,M_atp_m + M_dudp_m -> M_adp_m + M_dutp_m,r_4778
4891,R_r_4778,-1,enz_YKL067W,23.92,TurNuP predicted,6,2.7.4.6,metabolic,YKL067W,nucleoside diphosphate kinase,M_adp_m + M_dutp_m -> M_atp_m + M_dudp_m,r_4778


## Get the GPR relations from the TurnUp file

In [4]:
def create_gpr(row):
    """Create GPR relation for each enzyme."""
    genes = row['info_genes'].split(",")  # Split genes if there are multiple
    if len(genes) > 1:
        return f"({' and '.join(genes)})"  # Combine multiple genes with 'and'
    return genes[0]  # Single gene

turnup_gpr = turnup_kcats.copy()

# Apply to create GPR for each enzyme
turnup_gpr['enzyme_gpr'] = turnup_kcats.apply(create_gpr, axis=1)

# Group by reaction ID and combine enzyme GPRs with 'or'
turnup_gpr['GPR'] = turnup_gpr.groupby('rid')['enzyme_gpr'].transform(lambda x: ' or '.join(x))

# Drop duplicates to keep one row per reaction ID
turnup_gpr = turnup_gpr.drop_duplicates(subset=['rid'])[['rid', 'GPR']]

turnup_kcats_gpr = pd.merge(turnup_kcats,turnup_gpr, on='rid').explode('info_genes')
turnup_kcats_gpr

,rid,dirxn,enzyme,kcat_per_s,notes,info_active_sites,info_ecns,info_type,info_genes,info_name,info_reaction,model_rid,GPR
0,R_13BGHe,1,enz_YDR261C,13.67,TurNuP predicted,1,"3.2.1.58, 3.2.1.6",metabolic,YDR261C,"exo-1,3-beta-glucan glucohydrase",M_13BDglcn_e + M_h2o_e => M_glc__D_e,r_0370,YDR261C or YGR282C or YLR300W or YNR067C or YO...
1,R_13BGHe,1,enz_YGR282C,27.81,TurNuP predicted,1,"3.2.1.58, 3.2.1.6",metabolic,YGR282C,"exo-1,3-beta-glucan glucohydrase",M_13BDglcn_e + M_h2o_e => M_glc__D_e,r_0370,YDR261C or YGR282C or YLR300W or YNR067C or YO...
2,R_13BGHe,1,enz_YLR300W,25.83,TurNuP predicted,1,"3.2.1.58, 3.2.1.6",metabolic,YLR300W,"exo-1,3-beta-glucan glucohydrase",M_13BDglcn_e + M_h2o_e => M_glc__D_e,r_0370,YDR261C or YGR282C or YLR300W or YNR067C or YO...
3,R_13BGHe,1,enz_YNR067C,24.95,TurNuP predicted,1,"3.2.1.58, 3.2.1.6",metabolic,YNR067C,"exo-1,3-beta-glucan glucohydrase",M_13BDglcn_e + M_h2o_e => M_glc__D_e,r_0370,YDR261C or YGR282C or YLR300W or YNR067C or YO...
4,R_13BGHe,1,enz_YOR190W,24.61,TurNuP predicted,1,"3.2.1.58, 3.2.1.6",metabolic,YOR190W,"exo-1,3-beta-glucan glucohydrase",M_13BDglcn_e + M_h2o_e => M_glc__D_e,r_0370,YDR261C or YGR282C or YLR300W or YNR067C or YO...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4888,R_r_4777,1,enz_YKL067W,23.92,TurNuP predicted,6,2.7.4.6,metabolic,YKL067W,nucleoside diphosphate kinase,M_atp_m + M_didp_m -> M_adp_m + M_ditp_m,r_4777,YKL067W or YKL067W
4889,R_r_4777,-1,enz_YKL067W,23.92,TurNuP predicted,6,2.7.4.6,metabolic,YKL067W,nucleoside diphosphate kinase,M_adp_m + M_ditp_m -> M_atp_m + M_didp_m,r_4777,YKL067W or YKL067W
4890,R_r_4778,1,enz_YKL067W,23.92,TurNuP predicted,6,2.7.4.6,metabolic,YKL067W,nucleoside diphosphate kinase,M_atp_m + M_dudp_m -> M_adp_m + M_dutp_m,r_4778,YKL067W or YKL067W
4891,R_r_4778,-1,enz_YKL067W,23.92,TurNuP predicted,6,2.7.4.6,metabolic,YKL067W,nucleoside diphosphate kinase,M_adp_m + M_dutp_m -> M_atp_m + M_dudp_m,r_4778,YKL067W or YKL067W


## Map the TurnUp kcats to the uniprot information

In [5]:
# load the uniprot information
uniprot_df = pd.read_excel(UNIPROT_FILE_PATH)
#get the gene id from the gene names
uniprot_df['locus_tag'] = uniprot_df['Gene Names'].str.extract(locustag_regex)
uniprot_df = uniprot_df.drop('Gene Names', axis=1)
uniprot_df

,Entry,Length,Mass,Rhea ID,locus_tag
0,A0A0B7P3V8,1104,127189,RHEA:22508 RHEA:22508,YPL060C
1,A6ZM04,859,97628,RHEA:13065,NaN
2,A6ZU07,897,101717,RHEA:17989 RHEA:46608,NaN
3,A6ZXH8,997,115452,RHEA:38571 RHEA:38663 RHEA:38895 RHEA:67920,NaN
4,A6ZY89,1588,174807,RHEA:21968 RHEA:21096 RHEA:17737 RHEA:13121 RH...,NaN
...,...,...,...,...,...
48029,V9H1E1,26,3084,NaN,NaN
48030,V9H1G1,175,20671,NaN,NaN
48031,V9H1G4,26,3077,NaN,NaN
48032,V9H1L5,152,17301,NaN,NaN


In [6]:
#map the uniprot information to the turnup kcats
turnup_uniprot_gpr = pd.merge(turnup_kcats_gpr, uniprot_df[['locus_tag', 'Entry', 'Length', 'Mass']],
                             left_on ='info_genes', right_on ='locus_tag')
turnup_uniprot_gpr['direction']=turnup_uniprot_gpr['dirxn'].replace({1: 'f', -1: 'b'})

turnup_uniprot_gpr.rename({'kcat_per_s':'kcat_values', 
                           'model_rid':'rxn_id', 
                           'Entry':'enzyme_id',
                           'Mass': 'molMass', 
                           'rid': 'BiGG_id',
                           'locus_tag':'gene'
                          }, axis=1, inplace=True)
turnup_uniprot_gpr.drop(['enzyme', 'dirxn'], inplace=True, axis=1)
turnup_uniprot_gpr

,BiGG_id,kcat_values,notes,info_active_sites,info_ecns,info_type,info_genes,info_name,info_reaction,rxn_id,GPR,gene,enzyme_id,Length,molMass,direction
0,R_13BGHe,13.67,TurNuP predicted,1,"3.2.1.58, 3.2.1.6",metabolic,YDR261C,"exo-1,3-beta-glucan glucohydrase",M_13BDglcn_e + M_h2o_e => M_glc__D_e,r_0370,YDR261C or YGR282C or YLR300W or YNR067C or YO...,YDR261C,Q07793,1604,181661,f
1,R_13BGHe,13.67,TurNuP predicted,1,"3.2.1.58, 3.2.1.6",metabolic,YDR261C,"exo-1,3-beta-glucan glucohydrase",M_13BDglcn_e + M_h2o_e => M_glc__D_e,r_0370,YDR261C or YGR282C or YLR300W or YNR067C or YO...,YDR261C,O74302,440,49009,f
2,R_13BGHe,13.67,TurNuP predicted,1,"3.2.1.58, 3.2.1.6",metabolic,YDR261C,"exo-1,3-beta-glucan glucohydrase",M_13BDglcn_e + M_h2o_e => M_glc__D_e,r_0370,YDR261C or YGR282C or YLR300W or YNR067C or YO...,YDR261C,P52911,562,63508,f
3,R_13BGHe,27.81,TurNuP predicted,1,"3.2.1.58, 3.2.1.6",metabolic,YGR282C,"exo-1,3-beta-glucan glucohydrase",M_13BDglcn_e + M_h2o_e => M_glc__D_e,r_0370,YDR261C or YGR282C or YLR300W or YNR067C or YO...,YGR282C,P15703,313,34119,f
4,R_13BGHe,25.83,TurNuP predicted,1,"3.2.1.58, 3.2.1.6",metabolic,YLR300W,"exo-1,3-beta-glucan glucohydrase",M_13BDglcn_e + M_h2o_e => M_glc__D_e,r_0370,YDR261C or YGR282C or YLR300W or YNR067C or YO...,YLR300W,P23776,448,51311,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5295,R_r_4755,6.14,TurNuP predicted,1,1.1.99.40,metabolic,YEL071W,(R)-2-hydroxyglutarate:pyruvate oxidoreductase,M_pyr_m + M_2hog_m => M_lac__D_m + M_akg_m,r_4755,YDL178W or YEL071W,YEL071W,P39976,496,55225,f
5296,R_r_4757,64.85,TurNuP predicted,1,2.7.1.47,metabolic,YDR109C,D-ribulokinase,M_atp_c + M_s_4304_c -> M_adp_c + M_ru5p__D_c ...,r_4757,YDR109C or YDR109C,YDR109C,Q04585,715,79161,f
5297,R_r_4757,64.85,TurNuP predicted,1,2.7.1.47,metabolic,YDR109C,D-ribulokinase,M_atp_c + M_s_4304_c -> M_adp_c + M_ru5p__D_c ...,r_4757,YDR109C or YDR109C,YDR109C,G2WAL2,715,79105,f
5298,R_r_4757,64.85,TurNuP predicted,1,2.7.1.47,metabolic,YDR109C,D-ribulokinase,M_adp_c + M_ru5p__D_c + M_h_c -> M_atp_c + M_s...,r_4757,YDR109C or YDR109C,YDR109C,Q04585,715,79161,b


## Fill in missing kcats

In [7]:
#get information in the model
yeast_gem = cobra.io.read_sbml_model(YEAST_MODEL_FILE)

#get the file from GotEnzymes
got_enzymes_db = pd.read_excel(GotEnzymes_FILE_PATH).rename({'uniprot_id': 'enzyme_id'}, axis=1)

#How many reactions are not covered by TurnUp?
model_rxns_not_in_turnup = [rxn.id for rxn in yeast_gem.reactions if (
    len(rxn.genes)>0) and (rxn.id not in turnup_uniprot_gpr.rxn_id.to_list()
                          )]
print(f'There are {len(model_rxns_not_in_turnup)} enzymatic reactions in the model which are not covered by turnup')

#How many proteins are not covered by TurnUp?
gotenzymes_not_in_turnup = [gene for gene in got_enzymes_db.gene if gene not in turnup_uniprot_gpr.info_genes.to_list()]
print(f'There are {len(gotenzymes_not_in_turnup)} peptides in GotEnzymes which are not covered by turnup')

Set parameter Username
Academic license - for non-commercial use only - expires 2025-03-06
There are 202 enzymatic reactions in the model which are not covered by turnup
There are 692 peptides in GotEnzymes which are not covered by turnup


In [13]:
#Convert both dataframes such that each row represents a catalytical unit (each enzyme complex on a single row)
gene2protein = pd.Series(uniprot_df['Entry'].values,
                         index = uniprot_df['locus_tag']
                        ).to_dict()
got_enzymes_db_ec = merge_enzyme_complexes(got_enzymes_db, gene2protein)
turnup_uniprot_ec = merge_enzyme_complexes(turnup_uniprot_gpr, gene2protein)

,BiGG_id,kcat_values,notes,info_active_sites,info_ecns,info_type,info_genes,info_name,info_reaction,rxn_id,GPR,gene,enzyme_id,Length,molMass,direction
926,R_BTDD_RR,41.38,TurNuP predicted,2,1.1.1.4,metabolic,YAL060W,"(R,R)-butanediol dehydrogenase",M_btd_RR_c + M_nad_c -> M_actn__R_c + M_h_c + ...,r_0003,YAL060W or YAL060W,[YAL060W],P39714,382,41582,f
926,R_BTDD_RR,41.38,TurNuP predicted,2,1.1.1.4,metabolic,YAL060W,"(R,R)-butanediol dehydrogenase",M_btd_RR_c + M_nad_c -> M_actn__R_c + M_h_c + ...,r_0003,YAL060W or YAL060W,[YAL060W],P39714,382,41582,f
927,R_BTDD_RR,41.38,TurNuP predicted,2,1.1.1.4,metabolic,YAL060W,"(R,R)-butanediol dehydrogenase",M_actn__R_c + M_h_c + M_nadh_c -> M_btd_RR_c +...,r_0003,YAL060W or YAL060W,[YAL060W],P39714,382,41582,b
927,R_BTDD_RR,41.38,TurNuP predicted,2,1.1.1.4,metabolic,YAL060W,"(R,R)-butanediol dehydrogenase",M_actn__R_c + M_h_c + M_nadh_c -> M_btd_RR_c +...,r_0003,YAL060W or YAL060W,[YAL060W],P39714,382,41582,b
2830,R_r_0006,50.00,default for general-tx transport,1,NaN,transporter,YGR143W,"1,6-beta-glucan synthase",M_udpg_c => M_16BDglcn_ce + M_h_c + M_udp_c,r_0006,YGR143W or YPR159W,[YGR143W],P33336,771,86241,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2081,R_r_4777,23.92,TurNuP predicted,6,2.7.4.6,metabolic,YKL067W,nucleoside diphosphate kinase,M_adp_m + M_ditp_m -> M_atp_m + M_didp_m,r_4777,YKL067W or YKL067W,[YKL067W],P36010,153,17167,b
2082,R_r_4778,23.92,TurNuP predicted,6,2.7.4.6,metabolic,YKL067W,nucleoside diphosphate kinase,M_atp_m + M_dudp_m -> M_adp_m + M_dutp_m,r_4778,YKL067W or YKL067W,[YKL067W],P36010,153,17167,f
2082,R_r_4778,23.92,TurNuP predicted,6,2.7.4.6,metabolic,YKL067W,nucleoside diphosphate kinase,M_atp_m + M_dudp_m -> M_adp_m + M_dutp_m,r_4778,YKL067W or YKL067W,[YKL067W],P36010,153,17167,f
2083,R_r_4778,23.92,TurNuP predicted,6,2.7.4.6,metabolic,YKL067W,nucleoside diphosphate kinase,M_adp_m + M_dutp_m -> M_atp_m + M_dudp_m,r_4778,YKL067W or YKL067W,[YKL067W],P36010,153,17167,b


In [14]:
got_enzymes_db_ec

,rxn_id,bigg_id,kegg_id,Reactants,Products,EC,GPR,gene,enzyme_id,Length,molMass,kcat_values,direction
0,r_0001,D_LACDcm,R00197,"['C00256', 'C00125']","['C00126', 'C00022']",1.1.2.4,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,"[YDL174C, YEL039C]",P32891_P00044,587,65293,51.3780,f
0,r_0001,D_LACDcm,R00197,"['C00256', 'C00125']","['C00126', 'C00022']",1.1.2.4,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,"[YDL174C, YJR048W]",P32891_P00044,587,65293,51.3780,f
1,r_0001,D_LACDcm,R00197,"['C00256', 'C00125']","['C00126', 'C00022']",1.1.2.4,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,"[YDL174C, YEL039C]",P00045_P39976,113,12532,11.0000,f
1,r_0001,D_LACDcm,R00197,"['C00256', 'C00125']","['C00126', 'C00022']",1.1.2.4,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,"[YEL039C, YEL071W]",P00045_P39976,113,12532,11.0000,f
2,r_0001,D_LACDcm,R00197,"['C00256', 'C00125']","['C00126', 'C00022']",1.1.2.4,(YDL174C and YEL039C) or (YDL174C and YJR048W)...,"[YDL174C, YJR048W]",P32891_P00044,109,12182,11.0000,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4845,r_4776,NaN,R00139,"['C00002', 'C00104']","['C00008', 'C00081']",2.7.4.6,YKL067W,[YKL067W],P36010,153,17167,15.8160,b
4139,r_4777,NaN,R00139,"['C00002', 'C01344']","['C00008', 'C01345']",2.7.4.6,YKL067W,[YKL067W],P36010,153,17167,13.4934,f
4846,r_4777,NaN,R00139,"['C00002', 'C01344']","['C00008', 'C01345']",2.7.4.6,YKL067W,[YKL067W],P36010,153,17167,2.9067,b
4140,r_4778,NaN,R00139,"['C00002', 'C01346']","['C00008', 'C00460']",2.7.4.6,YKL067W,[YKL067W],P36010,153,17167,13.4934,f


In [15]:
# Add the rows to the dataframe which are not present in the TurnUp database
turnup_gotenzymes_merged = turnup_uniprot_ec.copy()

for i, row in got_enzymes_db_ec.iterrows():
    if (
    row.rxn_id not in turnup_uniprot_ec.rxn_id.to_list() or
    row.enzyme_id not in turnup_uniprot_ec.rxn_id.to_list()):
        new_row = row.copy().drop(['Reactants', 'Products','kegg_id'])
        new_row.rename({'bigg_id': 'BiGG_id', 'EC': 'info_ecns'})
        new_row['info_notes'] = 'GotEnzymes'
        new_row['info_genes'] = row.gene
        for col_name in ['info_active_sites','info_type' ,'info_name','info_reaction']:
            new_row[col_name] = ''
        turnup_gotenzymes_merged.loc[len(turnup_gotenzymes_merged)] = new_row
        
turnup_gotenzymes_merged

,BiGG_id,kcat_values,notes,info_active_sites,info_ecns,info_type,info_genes,info_name,info_reaction,rxn_id,GPR,gene,enzyme_id,Length,molMass,direction
926,R_BTDD_RR,41.3800,TurNuP predicted,2,1.1.1.4,metabolic,YAL060W,"(R,R)-butanediol dehydrogenase",M_btd_RR_c + M_nad_c -> M_actn__R_c + M_h_c + ...,r_0003,YAL060W or YAL060W,[YAL060W],P39714,382,41582,f
926,R_BTDD_RR,41.3800,TurNuP predicted,2,1.1.1.4,metabolic,YAL060W,"(R,R)-butanediol dehydrogenase",M_btd_RR_c + M_nad_c -> M_actn__R_c + M_h_c + ...,r_0003,YAL060W or YAL060W,[YAL060W],P39714,382,41582,f
927,R_BTDD_RR,41.3800,TurNuP predicted,2,1.1.1.4,metabolic,YAL060W,"(R,R)-butanediol dehydrogenase",M_actn__R_c + M_h_c + M_nadh_c -> M_btd_RR_c +...,r_0003,YAL060W or YAL060W,[YAL060W],P39714,382,41582,b
927,R_BTDD_RR,41.3800,TurNuP predicted,2,1.1.1.4,metabolic,YAL060W,"(R,R)-butanediol dehydrogenase",M_actn__R_c + M_h_c + M_nadh_c -> M_btd_RR_c +...,r_0003,YAL060W or YAL060W,[YAL060W],P39714,382,41582,b
2830,R_r_0006,50.0000,default for general-tx transport,1,NaN,transporter,YGR143W,"1,6-beta-glucan synthase",M_udpg_c => M_16BDglcn_ce + M_h_c + M_udp_c,r_0006,YGR143W or YPR159W,[YGR143W],P33336,771,86241,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12594,NaN,15.8160,NaN,,NaN,,[YKL067W],,,r_4776,YKL067W,[YKL067W],P36010,153,17167,b
12595,NaN,13.4934,NaN,,NaN,,[YKL067W],,,r_4777,YKL067W,[YKL067W],P36010,153,17167,f
12596,NaN,2.9067,NaN,,NaN,,[YKL067W],,,r_4777,YKL067W,[YKL067W],P36010,153,17167,b
12597,NaN,13.4934,NaN,,NaN,,[YKL067W],,,r_4778,YKL067W,[YKL067W],P36010,153,17167,f


## Save result to excel

In [16]:
# Get the information about the enzyme sectors
PAM_DATA_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_yeast9_EnzymaticData_240903.xlsx')
translational_protein_df = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = 'Translational')
unused_enzymes_df = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = 'UnusedEnzyme') #need to parse it later

In [17]:
# Save it to a new excel file
with pd.ExcelWriter(OUTPUT_FILE_PATH) as writer:
    turnup_gotenzymes_merged.to_excel(writer, sheet_name='ActiveEnzymes', index=False)
    translational_protein_df.to_excel(writer, sheet_name='Translational', index =False)
    unused_enzymes_df.to_excel(writer, sheet_name = 'UnusedEnzyme', index=False)